# Convert raw indoor measurements to long hourly format

Reshape the current upstream wide export into one sensor-variable-hour record per row. Calibration is a separate step.

**Step:** 01 (optional, upstream). Wide sensor export to long hourly data (folder `1_data_conversion`).

**Article scope:** the main article and Supplementary Material.

**Inputs:** The existing wide `data/indoor_all.csv` export.

**Outputs:** `data/indoor_all_long_hourly.csv` in the selected reproduction directory.

**How to use this notebook:** select the Python environment described in `README.md`, restart the kernel,
and run the cells from top to bottom. Edit the configuration cell before running.
The checks below stop execution when the current data no longer match the manuscript specification.

## Locate the code folder and load the shared functions

The repository is found by searching this notebook's folder and its parent folders, so Jupyter can be started anywhere inside the repository. Set `INDOOR_AQ_CODE` only when it starts elsewhere.
It loads only the cells tagged `library` in `00_shared_functions.ipynb` (its demonstration cell is not executed).
Each notebook has its own imports and configuration, so it never depends on another notebook's kernel state.

In [ ]:
# Setup: find the code folder and load the shared functions.
from pathlib import Path
import json
import os
import sys
import types


def locate_code_directory():
    """Locate this repository (the folder with 00_shared_functions.ipynb) from the working directory or INDOOR_AQ_CODE."""
    configured = os.environ.get("INDOOR_AQ_CODE")
    if configured:
        candidates = [Path(configured).expanduser().resolve()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if ((candidate / "00_shared_functions.ipynb").is_file()
                and (candidate / "5_did_cits").is_dir()):
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter in the repository folder, or set INDOOR_AQ_CODE to that folder.")

CODE = locate_code_directory()

# Load the cells tagged "library" from 00_shared_functions.ipynb as an in-memory
# module named `shared`, so that `from shared import ...` works below.
# No helper .py file is generated.
_library_path = CODE / "00_shared_functions.ipynb"
_library = types.ModuleType("shared")
_library.__file__ = str(_library_path)
_library.__dict__["CODE"] = CODE
for _cell in json.loads(_library_path.read_text(encoding="utf-8"))["cells"]:
    if _cell["cell_type"] == "code" and "library" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), str(_library_path), "exec"), _library.__dict__)
sys.modules["shared"] = _library

from shared import DATA, output_root
print(f"Manuscript code: {CODE}")

## Configure the output

Missing hours are represented by empty values on a complete UTC grid. This is structural reindexing, not measurement imputation.

In [ ]:
# Paths are shared by Python and R. Defaults: data/ for supplied inputs,
# results/ for this run. Set INDOOR_AQ_DATA and INDOOR_AQ_OUTPUT before
# starting Jupyter and R; relative paths are resolved against the repository.
OUTPUT = os.environ.get("INDOOR_AQ_OUTPUT") or None

## Set source paths and requested sensor IDs

The requested sensor list belongs to the upstream export and is broader than the final 23-home cohort. Residence screening occurs later in `03_prepare_cohort.ipynb`.

In [ ]:
import pandas as pd
from shared import DATA, output_root

SRC = DATA / 'indoor_all.csv'
OUT = output_root(OUTPUT) / 'data/indoor_all_long_hourly.csv'

# Sensor IDs requested from the upstream export (broader than the final
# 23-home cohort; residence screening happens in 03_prepare_cohort.ipynb).
KEEP = [18, 71, 17, 15, 28, 43, 158, 74, 9, 49, 66, 78, 88, 25, 38, 159, 160,
        27, 53, 48, 14, 30, 36, 104, 109, 19, 51, 153, 157, 20, 56, 16, 5, 60,
        57, 45, 2, 68, 52, 75, 91, 80, 81, 90, 95, 83]
KEEP_SET = set(KEEP)

## Read the wide export and locate measurement columns

Preserve the source text representation while selecting the PM2.5, CO2, temperature, and humidity columns by their existing prefixes.

In [ ]:
# --- read (values kept as strings to preserve original formatting) ----------
df = pd.read_csv(SRC, dtype=str, keep_default_na=False)
cols = df.columns.tolist()


def find(prefix):
    for c in cols:
        if c.startswith(prefix):
            return c
    raise KeyError(prefix)


c_pm, c_co2, c_temp, c_hum = (find("PM2.5"), find("CO2"),
                              find("Temperature"), find("Humidity"))

## Validate sensor identifiers

Extract numeric sensor suffixes and reject collisions in which one numeric ID maps to multiple location names. Report requested sensors missing from the source.

In [ ]:
# --- sensor_id + filter ------------------------------------------------------
df["sensor_id"] = df["Location Name"].str.extract(r"(\d+)$")[0].astype(int)

# guard: a kept numeric id must map to exactly one Location Name
dup = (df[df["sensor_id"].isin(KEEP_SET)]
       .groupby("sensor_id")["Location Name"].nunique())
clash = dup[dup > 1]
if len(clash):
    raise SystemExit(f"sensor_id collision(s): {clash.to_dict()}")

df = df[df["sensor_id"].isin(KEEP_SET)].copy()
found = sorted(df["sensor_id"].unique())
missing = sorted(KEEP_SET - set(found))
print(f"kept sensors: {len(found)} / {len(KEEP_SET)} requested")
if missing:
    print(f"  WARNING not found in source: {missing}")

## Reshape on a complete UTC hourly grid

Parse the UTC timestamp column before melting measurements into long format. Reindex every retained sensor and outcome to the same hourly grid. Duplicate source keys retain the first observation, matching the original converter.

In [ ]:
# --- UTC time ----------------------------------------------------------------
df["time_utc"] = pd.to_datetime(df["UTC Date/Time"], utc=True)

# --- wide -> long ------------------------------------------------------------
var_map = {c_co2: "co2", c_pm: "pm25", c_hum: "rh", c_temp: "temp"}
long = df.melt(id_vars=["sensor_id", "time_utc"], value_vars=list(var_map),
               var_name="variable", value_name="value")
long["variable"] = long["variable"].map(var_map)
long["value"] = long["value"].replace("", pd.NA)  # blank source cell -> NA

# --- reindex onto one complete UTC hourly grid -------------------------------
grid = pd.date_range(long["time_utc"].min(), long["time_utc"].max(),
                     freq="h", tz="UTC")
sensors = sorted(found, key=str)          # string order, like the reference file
variables = ["co2", "pm25", "rh", "temp"]
full_idx = pd.MultiIndex.from_product([sensors, variables, grid],
                                      names=["sensor_id", "variable", "time_utc"])

long = long.set_index(["sensor_id", "variable", "time_utc"])
long = long[~long.index.duplicated(keep="first")]   # drop any duplicate readings
out = long.reindex(full_idx).reset_index()

## Format local timestamps and save

Convert UTC timestamps to Europe/London only after constructing the regular grid; offsets preserve daylight saving transitions. Missing values remain blank. The output is ready for the saved-model calibration notebook.

In [ ]:
# --- format time (London local) + value -------------------------------------
out["time"] = out["time_utc"].dt.tz_convert("Europe/London").astype(str)
out["value"] = out["value"].fillna("")

out = out[["sensor_id", "variable", "time", "value"]]
out.to_csv(OUT, index=False, encoding="utf-8")

print(f"grid hours : {len(grid)}")
print(f"output rows: {len(out)}  -> {OUT}")
print(out.head(3).to_string(index=False))